# Analysing 10-K Reports of Apple, Microsoft and Tesla (2023-2025)

## BCGx Gen AI Job Simulation - Forage

### Builind a Rule based chatbot using Python (Pandas) and the SEC EDGAR database.

### - Vishal Srikanth

In [4]:
#Importing pandas library
import pandas as pd

In [10]:
#loading the CSV file into a DataFrame
df = pd.read_csv("Findings from 10-k reports.csv")
df.head(12)

,Company,Year,Total Revenue,Net Income,Total Assets,Total Liabilities,Cash Flow
0,Microsoft,2023,211915,72361,411976,205753,87582
1,Microsoft,2024,245122,88136,512163,243686,118548
2,Microsoft,2025,281724,101832,619003,275524,136162
3,Tesla,2023,96773,14974,106618,43009,13256
4,Tesla,2024,97690,7153,122070,48309,14923
5,Tesla,2025,94827,3855,137806,54941,14747
6,Apple,2023,383285,96995,352583,145308,110543
7,Apple,2024,391040,93736,364980,176392,118254
8,Apple,2025,416160,112010,359241,165361,111482


In [11]:
#Creating calculated year-over-year changes for each metric
df['Revenue Growth (%)'] = df.groupby(['Company'])['Total Revenue'].pct_change() * 100
df['Net Income Growth (%)'] = df.groupby(['Company'])['Net Income'].pct_change() * 100
df['Assets Growth (%)'] = df.groupby(['Company'])['Total Assets'].pct_change() * 100    
df['Liabilities Growth (%)'] = df.groupby(['Company'])['Total Liabilities'].pct_change() * 100
df['Cash Flow Growth (%)'] = df.groupby(['Company'])['Cash Flow'].pct_change() * 100

print(df[['Company', 'Revenue Growth (%)', 'Net Income Growth (%)', 'Assets Growth (%)', 'Liabilities Growth (%)', 'Cash Flow Growth (%)']])

     Company  Revenue Growth (%)  Net Income Growth (%)  Assets Growth (%)  \
0  Microsoft                 NaN                    NaN                NaN   
1  Microsoft           15.669962              21.800417          24.318650   
2  Microsoft           14.932156              15.539621          20.860546   
3      Tesla                 NaN                    NaN                NaN   
4      Tesla            0.947578             -52.230533          14.492862   
5      Tesla           -2.930699             -46.106529          12.890964   
6      Apple                 NaN                    NaN                NaN   
7      Apple            2.023299              -3.359967           3.516052   
8      Apple            6.423895              19.495178          -1.572415   

   Liabilities Growth (%)  Cash Flow Growth (%)  
0                     NaN                   NaN  
1               18.436183             35.356580  
2               13.065174             14.858117  
3                  

For the first row in every company the result in "NaN" because that refers to the year 2023, out starting year. Since its our starting year, we do not have any data for 2022, thus the change (%) cannot be computed. 

Aside from that, we can observe that excepts for Tesla in 2025, all the other companies grew every year in their revenue. 

Below is how a pro would've done it: 

In [13]:
# Fill NA values that result from pct_change calculations with 0 or an appropriate value
df.fillna(0, inplace=True)

# Display the dataframe to verify the calculations
print(df)

# Optionally, you could summarize these findings for each company
summary = df.groupby('Company').agg({
    'Revenue Growth (%)': 'mean',
    'Net Income Growth (%)': 'mean'
}).reset_index()

print("\nYear-over-Year Average Growth Rates (%):")
print(summary)

     Company  Year  Total Revenue  Net Income  Total Assets  \
0  Microsoft  2023         211915       72361        411976   
1  Microsoft  2024         245122       88136        512163   
2  Microsoft  2025         281724      101832        619003   
3      Tesla  2023          96773       14974        106618   
4      Tesla  2024          97690        7153        122070   
5      Tesla  2025          94827        3855        137806   
6      Apple  2023         383285       96995        352583   
7      Apple  2024         391040       93736        364980   
8      Apple  2025         416160      112010        359241   

   Total Liabilities  Cash Flow  Revenue Growth (%)  Net Income Growth (%)  \
0             205753      87582            0.000000               0.000000   
1             243686     118548           15.669962              21.800417   
2             275524     136162           14.932156              15.539621   
3              43009      13256            0.000000      

Logic Based Chatbot 

This chatbot uses predefined logic to asner user queries regarding financial data from the dataset above. 

The companies analysed are Apple, Microsoft and Tesla, for the years 2023, 2024 and 2025; and for the metrics: Total Revenue, Net Income, Total Assets, Total Liabilities, Cash Flow from Operating Activities, % Change in Revenue, % Change in Net Income, % Change in Total Assets, % Change in Total Liabilites, % Change in Cash Flow from operating activites over the years. 

By using an "If-Elif-Else" logic set, the chatbot provides responses to user queries based on predefined data mapping. 

First we develop the logic base, then the interactive Question-Answer feature either using text input or dropdown input.

In [39]:
# If-Elif-Else logic set for chatbot.
def simple_chatbot(user_query, company, year):
    """Hello! Welcome to the 10-K report analysis chatbot.

Try asking one of these questions for a company and an integer year:

    What is the total revenue?
    What is the net income?
    What are the total assets?
    What are the total liabilities?
    What is the cash flow from operating activities?
    How has total revenue changed over the last year?
    How has net income changed over the last year?
    How have total assets changed over the last year?
    How have total liabilities changed over the last year?
    How has cash flow from operating activities changed over the last year?
    """
    if user_query == "What is the total revenue?":
        metric = "Total Revenue"
    elif user_query == "What is the net income?":
        metric = "Net Income"
    elif user_query == "What are the total assets?":
        metric = "Total Assets"
    elif user_query == "What are the total liabilities?":
        metric = "Total Liabilities"
    elif user_query == "What is the cash flow from operating activities?":
        metric = "Cash Flow"
    elif user_query == "How has total revenue changed over the last year?":
        metric = "Revenue Growth (%)"
    elif user_query == "How has net income changed over the last year?":
        metric = "Net Income Growth (%)"
    elif user_query == "How have total assets changed over the last year?":
        metric = "Assets Growth (%)"
    elif user_query == "How have total liabilities changed over the last year?":
        metric = "Liabilities Growth (%)"
    elif user_query == "How has cash flow from operating activities changed over the last year?":
        metric = "Cash Flow Growth (%)"
    else:
        return "Sorry, I can only provide information on predefined queries."

    if not isinstance(year, int) or isinstance(year, bool):
        return "Please provide the year as an integer, such as 2025."

    company_rows = df[df["Company"].str.casefold() == company.strip().casefold()]
    rows = company_rows[company_rows["Year"] == year]
    if rows.empty:
        return f"No data found for {company} in {year}."
    if len(rows) != 1:
        return f"Multiple records found for {company} in {year}; cannot determine the answer."

    row = rows.iloc[0]
    if metric.endswith("Growth (%)"):
        previous_rows = company_rows[company_rows["Year"] == year - 1]
        if len(previous_rows) != 1 or pd.isna(row[metric]):
            return f"Year-over-year growth is unavailable for {row['Company']} in {year}."
        return f"{row['Company']}'s {metric} in {year} was {row[metric]:.2f}%."

    if pd.isna(row[metric]):
        return f"{metric} is unavailable for {row['Company']} in {year}."
    return f"{row['Company']}'s {metric} in {year} was {row[metric]:,.2f} (USD)."

In [41]:
print(simple_chatbot.__doc__)
question = input("Ask me a question like the ones as listed above: ").strip()
company = input("Enter a company (Apple, Microsoft, or Tesla): ").strip()
year_text = input("Enter a year (2023, 2024, or 2025): ").strip()
try:
    year = int(year_text)
except ValueError:
    print("Please provide the year as a whole number, such as 2025.")
else:
    print(simple_chatbot(question, company, year))

Hello! Welcome to the 10-K report analysis chatbot.

Try asking one of these questions for a company and an integer year:

    What is the total revenue?
    What is the net income?
    What are the total assets?
    What are the total liabilities?
    What is the cash flow from operating activities?
    How has total revenue changed over the last year?
    How has net income changed over the last year?
    How have total assets changed over the last year?
    How have total liabilities changed over the last year?
    How has cash flow from operating activities changed over the last year?
    
Apple's Net Income Growth (%) in 2024 was -3.36%.


### Chatbot Documentation

The user runs the last function and sees an introductory message and a list a accetable questions before being prompted to type in their questions in the input bar on top.

The chatbot makes things interactive by greeeting the user upon activation, with this message: "Hello! Welcome to the 10-K report analysis chatbot."

Following which, it introduces and lists the set of predefined questions the user can ask with a friendly message:
"Try asking one of these questions for a company and an integer year:

    What is the total revenue?
    What is the net income?
    What are the total assets?
    What are the total liabilities?
    What is the cash flow from operating activities?
    How has total revenue changed over the last year?
    How has net income changed over the last year?
    How have total assets changed over the last year?
    How have total liabilities changed over the last year?
    How has cash flow from operating activities changed over the last year?"
    
For all acceptable questions, the chatbot returns an answer with the appropriate units (USD for numerical values and % for change percentage). For queries that are out of the bounds of the chatbot, it returns the following messages:

- Non predefined question: "Sorry, I can only provide information on predefined queries."

- Non predefined year or metric: "{metric} is unavailable for {row[Company]} in {year}."

Response Handling in case of Data Inconsistency:

In cases where there is data inconsistency, such as empty rows or multiple values per row, the chatbot returns the message "No data found for {company} in {year}." or "Multiple records found for {company} in {year}; cannot determine the answer." Another important feature was % change for the first year of data. Since we do not have data for 2022, the % change columns for 2023 will be blank, thus the chatbot returns "Year-over-year growth is unavailable for {row['Company']} in {year}."

